In [1]:
import pandas as pd

In [2]:
from data.nbaDataImport import load_nba_games

In [3]:
nba_data = load_nba_games()

         id   game_date  season  home_team_id         home_team_name  \
0  18446819  2025-10-21    2025            21  Oklahoma City Thunder   
1  18446820  2025-10-21    2025            14     Los Angeles Lakers   
2  18446821  2025-10-22    2025            20        New York Knicks   
3  18446822  2025-10-22    2025             7       Dallas Mavericks   
4  18446823  2025-10-22    2025             4      Charlotte Hornets   

   visitor_team_id      visitor_team_name  home_team_score  \
0               11        Houston Rockets              125   
1               10  Golden State Warriors              109   
2                6    Cleveland Cavaliers              119   
3               27      San Antonio Spurs               92   
4                3          Brooklyn Nets              136   

   visitor_team_score  postseason  
0                 124       False  
1                 119       False  
2                 111       False  
3                 125       False  
4             

In [4]:
nba_data.head

<bound method NDFrame.head of             id   game_date  season  home_team_id         home_team_name  \
0     18446819  2025-10-21    2025            21  Oklahoma City Thunder   
1     18446820  2025-10-21    2025            14     Los Angeles Lakers   
2     18446821  2025-10-22    2025            20        New York Knicks   
3     18446822  2025-10-22    2025             7       Dallas Mavericks   
4     18446823  2025-10-22    2025             4      Charlotte Hornets   
...        ...         ...     ...           ...                    ...   
2638  18444560  2025-06-11    2024            12         Indiana Pacers   
2639  18444561  2025-06-13    2024            12         Indiana Pacers   
2640  18444562  2025-06-16    2024            21  Oklahoma City Thunder   
2641  18444563  2025-06-19    2024            12         Indiana Pacers   
2642  18444564  2025-06-22    2024            21  Oklahoma City Thunder   

      visitor_team_id      visitor_team_name  home_team_score  \
0   

In [5]:
nba_data.describe()

,id,season,home_team_id,visitor_team_id,home_team_score,visitor_team_score
count,2.643000e+03,2643.000000,2643.000000,2643.000000,2643.000000,2643.000000
mean,1.740806e+07,2024.500189,15.491487,15.479001,115.199773,113.377601
std,1.505938e+06,0.500095,8.570599,8.575434,12.840466,12.943088
min,1.590744e+07,2024.000000,1.000000,1.000000,77.000000,66.000000
25%,1.590810e+07,2024.000000,8.000000,8.000000,106.000000,105.000000
50%,1.844682e+07,2025.000000,16.000000,16.000000,115.000000,113.000000
75%,1.844748e+07,2025.000000,23.000000,23.000000,124.000000,122.000000
max,2.171614e+07,2025.000000,30.000000,30.000000,157.000000,162.000000


In [6]:
nba_data.columns

Index(['id', 'game_date', 'season', 'home_team_id', 'home_team_name',
       'visitor_team_id', 'visitor_team_name', 'home_team_score',
       'visitor_team_score', 'postseason'],
      dtype='str')

In [7]:
# Now that we have the nba data imported and have inspected the data a bit above, I want to transform what I have into data that can better serve game prediction

In [8]:
# I'd like to create four several fields, including:

In [9]:
# * season_record

In [10]:
# * record_last_five_games

In [11]:
# * offensive_rating

In [12]:
# * offensive_rating_last_five_games

In [13]:
# * defensive_rating

In [14]:
# * defensive_rating_last_five_games

In [15]:
# * rest_days

In [16]:
#start by stacking home and away results under one single "team_id"

In [20]:
# rename game_date column to date for readability
nba_data = nba_data.rename(columns={"game_date":"date"})

# One row per game from the home team's perspective.
home = nba_data[
    ["id", "date", "season", "home_team_id",
     "home_team_score", "visitor_team_score"]
].rename(columns={
    "home_team_id": "team_id",
    "home_team_score": "points_for",
    "visitor_team_score": "points_against"
})

# The same games from the visiting team's perspective.
away = nba_data[
    ["id", "date", "season", "visitor_team_id",
     "visitor_team_score", "home_team_score"]
].rename(columns={
    "visitor_team_id": "team_id",
    "visitor_team_score": "points_for",
    "home_team_score": "points_against"
})

# Stack the tables vertically, similar to SQL UNION ALL.
team_games = pd.concat([home, away], ignore_index=True)

team_games["date"] = pd.to_datetime(team_games["date"], utc=True)

team_games = team_games.sort_values(
    ["team_id", "season", "date", "id"]
).reset_index(drop=True)

In [ ]:
# Create a win indicator

In [21]:
team_games["won"] = (
    team_games["points_for"] > team_games["points_against"]
).astype(int)

team_games["previous_game_won"] = (
    team_games
    .groupby(["team_id", "season"])["won"]
    .shift(1)
)

In [23]:
team_games.head()

,id,date,season,team_id,points_for,points_against,won,previous_game_won
0,15907441,2024-10-23 00:00:00+00:00,2024,1,120,116,1,NaN
1,15907456,2024-10-25 00:00:00+00:00,2024,1,125,120,1,1.0
2,15907477,2024-10-27 00:00:00+00:00,2024,1,104,128,0,1.0
3,15907480,2024-10-28 00:00:00+00:00,2024,1,119,121,0,0.0
4,15907498,2024-10-30 00:00:00+00:00,2024,1,120,133,0,0.0


In [24]:
# inspect one team

In [25]:
team_games.loc[
    team_games["team_id"] == team_games["team_id"].iloc[0],
    ["date", "season", "won", "previous_game_won"]
].head(10)

,date,season,won,previous_game_won
0,2024-10-23 00:00:00+00:00,2024,1,NaN
1,2024-10-25 00:00:00+00:00,2024,1,1.0
2,2024-10-27 00:00:00+00:00,2024,0,1.0
3,2024-10-28 00:00:00+00:00,2024,0,0.0
4,2024-10-30 00:00:00+00:00,2024,0,0.0
5,2024-11-01 00:00:00+00:00,2024,0,0.0
6,2024-11-03 00:00:00+00:00,2024,1,0.0
7,2024-11-04 00:00:00+00:00,2024,0,1.0
8,2024-11-06 00:00:00+00:00,2024,1,0.0
9,2024-11-08 00:00:00+00:00,2024,0,1.0


In [26]:
# transform game data to include points differetnial and rest days

In [27]:
team_games["date"] = pd.to_datetime(team_games["date"], utc=True)
team_games = team_games.sort_values(
    ["team_id", "season", "date", "id"]
).reset_index(drop=True)

team_games["point_differential"] = (
    team_games["points_for"] - team_games["points_against"]
)

groups = team_games.groupby(["team_id", "season"])

#win percentages BEFORE this game.
groups = team_games.groupby(["team_id", "season"])

team_games["season_win_percentage"] = groups["won"].transform(
    lambda s: s.shift(1).expanding().mean()
)

team_games["win_percentage_last_five_games"] = groups["won"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=5).mean()
)

# Calendar days between game dates: back-to-back games = 1.
previous_date = groups["date"].shift(1)
team_games["rest_days_since_last_game"] = (
    team_games["date"].dt.normalize() -
    previous_date.dt.normalize()
).dt.days

# Average scoring margin over ALL earlier games this season.
team_games["average_point_differential"] = (
    groups["point_differential"].transform(
        lambda s: s.shift(1).expanding().mean()
    )
)

In [28]:
team_games.head()

,id,date,season,team_id,points_for,points_against,won,previous_game_won,point_differential,season_win_percentage,win_percentage_last_five_games,rest_days_since_last_game,average_point_differential
0,15907441,2024-10-23 00:00:00+00:00,2024,1,120,116,1,NaN,4,NaN,NaN,NaN,NaN
1,15907456,2024-10-25 00:00:00+00:00,2024,1,125,120,1,1.0,5,1.000000,NaN,2.0,4.00
2,15907477,2024-10-27 00:00:00+00:00,2024,1,104,128,0,1.0,-24,1.000000,NaN,2.0,4.50
3,15907480,2024-10-28 00:00:00+00:00,2024,1,119,121,0,0.0,-2,0.666667,NaN,1.0,-5.00
4,15907498,2024-10-30 00:00:00+00:00,2024,1,120,133,0,0.0,-13,0.500000,NaN,2.0,-4.25


In [34]:
# Add scoring averages, since off and def ratings impossible to calc w/o possession count

In [35]:
for source, prefix in [
    ("points_for", "points_scored"),
    ("points_against", "points_allowed"),
]:
    team_games[f"{prefix}_per_game"] = groups[source].transform(
        lambda s: s.shift(1).expanding().mean()
    )
    team_games[f"{prefix}_last_five_games"] = groups[source].transform(
        lambda s: s.shift(1).rolling(5, min_periods=5).mean()
    )

In [37]:
team_games.head()

,id,date,season,team_id,points_for,points_against,won,previous_game_won,point_differential,season_win_percentage,win_percentage_last_five_games,rest_days_since_last_game,average_point_differential,points_scored_per_game,points_scored_last_five_games,points_allowed_per_game,points_allowed_last_five_games
0,15907441,2024-10-23 00:00:00+00:00,2024,1,120,116,1,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,15907456,2024-10-25 00:00:00+00:00,2024,1,125,120,1,1.0,5,1.000000,NaN,2.0,4.00,120.000000,NaN,116.000000,NaN
2,15907477,2024-10-27 00:00:00+00:00,2024,1,104,128,0,1.0,-24,1.000000,NaN,2.0,4.50,122.500000,NaN,118.000000,NaN
3,15907480,2024-10-28 00:00:00+00:00,2024,1,119,121,0,0.0,-2,0.666667,NaN,1.0,-5.00,116.333333,NaN,121.333333,NaN
4,15907498,2024-10-30 00:00:00+00:00,2024,1,120,133,0,0.0,-13,0.500000,NaN,2.0,-4.25,117.000000,NaN,121.250000,NaN


In [30]:
# now join the transformed data 'team_games' back to the original table 'nba_data' above

In [40]:
features = [
    "season_win_percentage",
    "win_percentage_last_five_games",
    "rest_days_since_last_game",
    "average_point_differential",
    "points_scored_per_game",
    "points_scored_last_five_games",
    "points_allowed_per_game",
    "points_allowed_last_five_games"
]

# Select only identifiers and pregame features.
# Exclude current-game scores/results to avoid leakage.
team_features = team_games[["id", "team_id"] + features].copy()

home_features = team_features.rename(columns={
    "team_id": "home_team_id",
    **{column: f"home_{column}" for column in features},
})

visitor_features = team_features.rename(columns={
    "team_id": "visitor_team_id",
    **{column: f"visitor_{column}" for column in features},
})

nba_model_data = nba_data.merge(
    home_features,
    on=["id", "home_team_id"],
    how="left",
    validate="one_to_one",
).merge(
    visitor_features,
    on=["id", "visitor_team_id"],
    how="left",
    validate="one_to_one",
)

In [41]:
# Now back to previous lesson ...

In [42]:
#... first setting up a target for prediction (y)

In [43]:
home_team = 0

In [46]:
nba_model_data["home_win"] = (nba_data["home_team_score"] > nba_data["visitor_team_score"]).astype(int)

In [47]:
nba_model_data.head()

,id,date,season,home_team_id,home_team_name,visitor_team_id,visitor_team_name,home_team_score,visitor_team_score,postseason,...,home_points_allowed_last_five_games,visitor_season_win_percentage,visitor_win_percentage_last_five_games,visitor_rest_days_since_last_game,visitor_average_point_differential,visitor_points_scored_per_game,visitor_points_scored_last_five_games,visitor_points_allowed_per_game,visitor_points_allowed_last_five_games,home_win
0,18446819,2025-10-21,2025,21,Oklahoma City Thunder,11,Houston Rockets,125,124,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,18446820,2025-10-21,2025,14,Los Angeles Lakers,10,Golden State Warriors,109,119,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,18446821,2025-10-22,2025,20,New York Knicks,6,Cleveland Cavaliers,119,111,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,18446822,2025-10-22,2025,7,Dallas Mavericks,27,San Antonio Spurs,92,125,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,18446823,2025-10-22,2025,4,Charlotte Hornets,3,Brooklyn Nets,136,117,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


In [48]:
y = nba_model_data["home_win"]

In [55]:
#now add the new features and assign them to X

In [56]:
#first show column names

In [57]:
nba_model_data.columns

Index(['id', 'date', 'season', 'home_team_id', 'home_team_name',
       'visitor_team_id', 'visitor_team_name', 'home_team_score',
       'visitor_team_score', 'postseason', 'home_season_win_percentage',
       'home_win_percentage_last_five_games', 'home_rest_days_since_last_game',
       'home_average_point_differential', 'home_points_scored_per_game',
       'home_points_scored_last_five_games', 'home_points_allowed_per_game',
       'home_points_allowed_last_five_games', 'visitor_season_win_percentage',
       'visitor_win_percentage_last_five_games',
       'visitor_rest_days_since_last_game',
       'visitor_average_point_differential', 'visitor_points_scored_per_game',
       'visitor_points_scored_last_five_games',
       'visitor_points_allowed_per_game',
       'visitor_points_allowed_last_five_games', 'home_win'],
      dtype='str')

In [58]:
nba_model_features = [
       'home_season_win_percentage',
       'home_win_percentage_last_five_games', 'home_rest_days_since_last_game',
       'home_average_point_differential', 'home_points_scored_per_game',
       'home_points_scored_last_five_games', 'home_points_allowed_per_game',
       'home_points_allowed_last_five_games', 'visitor_season_win_percentage',
       'visitor_win_percentage_last_five_games',
       'visitor_rest_days_since_last_game',
       'visitor_average_point_differential', 'visitor_points_scored_per_game',
       'visitor_points_scored_last_five_games',
       'visitor_points_allowed_per_game',
       'visitor_points_allowed_last_five_games',
] 

In [59]:
X = nba_model_data[nba_model_features]

In [60]:
X.describe()

,home_season_win_percentage,home_win_percentage_last_five_games,home_rest_days_since_last_game,home_average_point_differential,home_points_scored_per_game,home_points_scored_last_five_games,home_points_allowed_per_game,home_points_allowed_last_five_games,visitor_season_win_percentage,visitor_win_percentage_last_five_games,visitor_rest_days_since_last_game,visitor_average_point_differential,visitor_points_scored_per_game,visitor_points_scored_last_five_games,visitor_points_allowed_per_game,visitor_points_allowed_last_five_games
count,2612.000000,2494.000000,2612.000000,2612.000000,2612.000000,2494.000000,2612.000000,2494.000000,2614.000000,2492.000000,2614.000000,2614.000000,2614.000000,2492.000000,2614.000000,2492.000000
mean,0.509230,0.508581,2.202144,0.294589,114.840824,114.436808,114.546235,114.099198,0.508477,0.507785,2.136572,0.340196,114.931709,114.535795,114.591513,114.253852
std,0.187153,0.267295,1.101385,6.625959,4.657163,6.941223,5.035321,7.304937,0.189404,0.264999,0.989854,6.551590,4.704258,6.772826,4.963606,7.361829
min,0.000000,0.000000,1.000000,-36.000000,87.000000,89.600000,91.000000,90.200000,0.000000,0.000000,1.000000,-21.500000,95.500000,94.600000,87.000000,89.400000
25%,0.389775,0.400000,2.000000,-4.000000,111.854722,109.600000,111.214286,109.200000,0.380048,0.400000,2.000000,-4.000000,111.826087,110.000000,111.316009,109.200000
50%,0.529412,0.600000,2.000000,0.925824,115.137255,114.400000,114.503086,113.800000,0.529412,0.600000,2.000000,0.963366,115.138494,114.600000,114.622845,114.200000
75%,0.625343,0.800000,2.000000,4.500000,117.850028,119.400000,117.550179,119.000000,0.626816,0.800000,2.000000,4.535507,117.903917,119.200000,117.545455,119.000000
max,1.000000,1.000000,9.000000,38.500000,138.000000,135.200000,140.000000,138.800000,1.000000,1.000000,9.000000,36.000000,140.000000,135.000000,141.000000,140.200000


In [61]:
# drop NaNs

In [62]:
model_data = nba_model_data.dropna(subset=nba_model_features).copy()

In [67]:
model_data[nba_model_features].isna().sum()

home_season_win_percentage                0
home_win_percentage_last_five_games       0
home_rest_days_since_last_game            0
home_average_point_differential           0
home_points_scored_per_game               0
home_points_scored_last_five_games        0
home_points_allowed_per_game              0
home_points_allowed_last_five_games       0
visitor_season_win_percentage             0
visitor_win_percentage_last_five_games    0
visitor_rest_days_since_last_game         0
visitor_average_point_differential        0
visitor_points_scored_per_game            0
visitor_points_scored_last_five_games     0
visitor_points_allowed_per_game           0
visitor_points_allowed_last_five_games    0
dtype: int64

In [68]:
# follow last week's lesson, except split the data into train and validation, per Kaggle's lessons

In [69]:
train = model_data[model_data["season"] == 2024]
validation = model_data[model_data["season"] == 2025]

X_train = train[nba_model_features]
y_train = train["home_win"]

X_validation = validation[nba_model_features]
y_validation = validation["home_win"]

In [70]:
#make the model

In [71]:
from sklearn.tree import DecisionTreeRegressor

In [72]:
nba_model = DecisionTreeRegressor()

In [73]:
nba_model.fit(X_train, y_train)

,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree wi

In [74]:
home_win_predictions = nba_model.predict(X_validation)

In [75]:
#now use MAE as metric

In [77]:
from sklearn.metrics import mean_absolute_error

In [78]:
print(mean_absolute_error(y_validation, home_win_predictions))

0.46983105390185037


In [80]:
from sklearn.metrics import accuracy_score

predictions = nba_model.predict(X_validation)

model_accuracy = accuracy_score(y_validation, predictions)
baseline_accuracy = y_validation.mean()

print(f"Decision tree accuracy: {model_accuracy:.2%}")
print(f"Always-home-win baseline: {baseline_accuracy:.2%}")

Decision tree accuracy: 53.02%
Always-home-win baseline: 55.27%


In [81]:
# eval: Model is underperforming the baseline

In [82]:
for name, frame in [
    ("Training", train),
    ("Validation", validation),
]:
    print(
        name,
        "Games:", len(frame),
        "Dates:", frame["date"].min(), "to", frame["date"].max(),
        "Seasons:", frame["season"].unique(),
    )

print(
    "Shared game IDs:",
    len(set(train["id"]) & set(validation["id"]))
)

Training Games: 1243 Dates: 2024-11-01 to 2025-06-22 Seasons: [2024]
Validation Games: 1243 Dates: 2025-10-31 to 2026-06-13 Seasons: [2025]
Shared game IDs: 0


In [84]:
print(
    "Training accuracy:",
    f"{nba_model.score(X_train, y_train):.2%}"
)
print(
    "Validation accuracy:",
    f"{nba_model.score(X_validation, y_validation):.2%}"
)
print(nba_model.get_params())

Training accuracy: 100.00%
Validation accuracy: -90.04%
{'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': None, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': None, 'splitter': 'best'}


In [85]:
print(type(nba_model))

<class 'sklearn.tree._classes.DecisionTreeRegressor'>


In [87]:
#NOte: AT this point, I realized I was using the DecisionTreeRegressor, instead of the DecisionTreeClassifer!

In [ ]:
#swapping in the new model here:

In [86]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=30,
    random_state=42,
)

model.fit(X_train, y_train)

print(
    "Training accuracy:",
    f"{accuracy_score(y_train, model.predict(X_train)):.2%}"
)
print(
    "Validation accuracy:",
    f"{accuracy_score(y_validation, model.predict(X_validation)):.2%}"
)
print("Always-home baseline:", f"{y_validation.mean():.2%}")

Training accuracy: 65.97%
Validation accuracy: 62.51%
Always-home baseline: 55.27%


In [92]:
# Success!!
# Beating baseline by ~7 percentage points
#...bringing back MAE since that's what's in the Kaggle lesson:

In [90]:
home_win_predictions = model.predict(X_validation)

In [91]:
print(mean_absolute_error(y_validation, home_win_predictions))

0.37489943684633953
